# Two Asset SJM baseline

This notebook runs a two-asset statistical jump-model baseline: fit equity and bond regimes, convert selected regimes into allocation signals, backtest baseline strategies, and summarize the results.

In [ ]:
from pathlib import Path

import warnings

import matplotlib.pyplot as plt
import pandas as pd

from functions import (
    DEFAULT_CHOSEN_LAMBDAS,
    DEFAULT_LAMBDA_GRID,
    AllocationConfig,
    build_allocation_signals,
    build_allocation_weights,
    build_feature_panel,
    fit_asset_sjm,
    fit_regime_labels,
    load_panel,
    plot_backtest,
    run_backtest,
    summarize_backtest,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.options.display.float_format = "{:.6f}".format

project_dir = Path.cwd()
data_dir = project_dir / "data"
output_dir = project_dir / "outputs"
data_path = data_dir / "model_panel_start_1970_01_31.parquet"
output_dir.mkdir(exist_ok=True)

In [ ]:
panel_raw, panel = load_panel(data_path)

In [ ]:
input_summary = pd.DataFrame(
    {
        "rows": [len(panel)],
        "start_date": [panel["date"].min().date()],
        "end_date": [panel["date"].max().date()],
        "columns": [len(panel.columns)],
    }
)
input_summary

In [ ]:
feature_panel = build_feature_panel(panel)
feature_panel.filter(regex="^(date|eq_|bd_|macro_|eq_bd_corr)").tail()

In [ ]:
equity_feature_cols = [
    "eq_logdd_h1", "eq_logdd_h4",
    "eq_mean_h1", "eq_mean_h2", "eq_mean_h4",
    "eq_sortino_h1", "eq_sortino_h2", "eq_sortino_h4",
]

equity_diagnostics, equity_labels = fit_asset_sjm(
    asset="equity",
    feature_panel=feature_panel,
    excess_col="equity_excess",
    feature_cols=equity_feature_cols,
    lambda_grid=DEFAULT_LAMBDA_GRID,
)
equity_diagnostics

In [ ]:
bond_feature_cols = [
    "bd_mean_h1", "bd_mean_h2", "bd_mean_h4",
    "bd_sharpe_h1", "bd_sharpe_h2", "bd_sharpe_h4",
]

bond_diagnostics, bond_labels = fit_asset_sjm(
    asset="bond",
    feature_panel=feature_panel,
    excess_col="bond_excess",
    feature_cols=bond_feature_cols,
    lambda_grid=DEFAULT_LAMBDA_GRID,
)
bond_diagnostics

In [ ]:
sjm_diagnostics, sjm_labels = fit_regime_labels(feature_panel, lambda_grid=DEFAULT_LAMBDA_GRID)
selected_diagnostics = sjm_diagnostics.loc[
    sjm_diagnostics.apply(lambda row: row["lambda"] == DEFAULT_CHOSEN_LAMBDAS[row["asset"]], axis=1)
].reset_index(drop=True)
selected_diagnostics

In [ ]:
config = AllocationConfig()
alloc_panel, history_panel = build_allocation_signals(
    panel=panel,
    labels=sjm_labels,
    chosen_lambdas=DEFAULT_CHOSEN_LAMBDAS,
    min_history_months=config.min_history_months,
)
alloc_panel.head()

In [ ]:
weights = build_allocation_weights(
    alloc_panel=alloc_panel,
    history_panel=history_panel,
    config=config,
)
weights.head()

In [ ]:
backtest = run_backtest(
    alloc_panel=alloc_panel,
    weights=weights,
    config=config,
)
backtest.head()

In [ ]:
summary = summarize_backtest(backtest)
summary

In [ ]:
fig, axes = plot_backtest(backtest, summary)
plt.show()